# Camada Silver — State of Data Brasil (edição 2024/2025)

Dono: Maycon

Responsabilidade desta etapa: limpar e padronizar a Bronze.

1. Seleciona e renomeia só as 174 colunas relevantes (dicionário em `utils/constants.py`)
2. Converte colunas de múltipla escolha para boolean, preservando null = pergunta-portão
3. Adiciona a coluna `edicao` (necessária pra unir com as outras 2 bases depois)
4. Remove duplicidade de `token`
5. Grava Parquet final, pronto pro Athena consultar

**Decisão de limpeza de nulos:** ao contrário do que o Carlos fez na base 2023
(preencher nulo com "Sem resposta"), aqui optei por **manter os nulos como
null**. A maior parte dos nulos desta pesquisa é estrutural — pergunta que
nunca apareceu pra aquela pessoa por causa de uma pergunta-portão anterior
(`atua_como_gestor`, `situação de trabalho ativa` etc), não erro de coleta.
Preencher esses casos com um valor fixo distorceria os `GROUP BY` (contaria
"Sem resposta" como se fosse uma categoria de resposta real). O mapeamento
completo de qual coluna é condicionada a qual pergunta está documentado em
`utils/constants.py` → `REGRAS_DE_NULO`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

from utils.config import get_spark_session
from utils.functions import (
    renomear_e_filtrar_colunas, colunas_multipla_escolha_para_boolean,
    adicionar_coluna_edicao, relatorio_nulos
)
from utils.constants import (
    BRONZE_PATH, SILVER_PATH, EDICAO, RENAME_COLUNAS,
    COLUNAS_MULTIPLA_ESCOLHA, COLUNAS_JA_BOOLEANAS, REGRAS_DE_NULO
)

spark = get_spark_session()

df_bronze = spark.read.parquet(BRONZE_PATH)
print("Bronze lida:", df_bronze.count(), "linhas,", len(df_bronze.columns), "colunas")

26/08/07 19:22:56 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/07 19:22:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/07 19:22:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Bronze lida: 5217 linhas, 403 colunas


## 1. Seleção e renomeação das colunas relevantes

In [2]:
df = renomear_e_filtrar_colunas(df_bronze, RENAME_COLUNAS)
print("Após filtro/rename:", len(df.columns), "colunas (de", len(RENAME_COLUNAS), "no dicionário)")

Após filtro/rename: 174 colunas (de 174 no dicionário)


## 2. Tipagem booleana

Colunas de múltipla escolha (0.0/1.0/NaN) viram boolean. Colunas que já chegam boolean (`atua_como_gestor`, `satisfeito`) só reforçam o tipo.

In [3]:
df = colunas_multipla_escolha_para_boolean(df, COLUNAS_MULTIPLA_ESCOLHA)

for c in COLUNAS_JA_BOOLEANAS:
    df = df.withColumn(c, df[c].cast("boolean"))

print(f"{len(COLUNAS_MULTIPLA_ESCOLHA)} colunas de múltipla escolha convertidas para boolean")

131 colunas de múltipla escolha convertidas para boolean


## 3. Coluna de edição (pra unir com 2023_2024 e 2025_2026 depois)

In [4]:
df = adicionar_coluna_edicao(df, EDICAO)

## 4. Remoção de duplicidade por token

In [5]:
antes = df.count()
df = df.dropDuplicates(["token"])
depois = df.count()
print(f"Linhas antes: {antes} | depois: {depois} | duplicados removidos: {antes - depois}")

Linhas antes: 5217 | depois: 5215 | duplicados removidos: 2


## 5. Conferência dos nulos estruturais

Validar que o percentual de nulo das colunas listadas em `REGRAS_DE_NULO` está de fato alto (confirma que é pergunta-portão, não erro).

In [6]:
relatorio = relatorio_nulos(df)
relatorio.show(15, truncate=False)

26/08/07 19:23:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------------------+---------+---------+
|coluna                     |qtd_nulos|pct_nulos|
+---------------------------+---------+---------+
|oportunidade_buscada       |4929     |94.5     |
|exp_processos_seletivos    |4926     |94.5     |
|tempo_busca_oportunidade   |4925     |94.4     |
|objetivo_dados             |4674     |89.6     |
|ferramentas_qualidade_dados|4656     |89.3     |
|tech_data_warehouse        |4490     |86.1     |
|tech_data_lake             |4481     |85.9     |
|possui_data_warehouse      |4299     |82.4     |
|possui_data_lake           |4293     |82.3     |
|etl_de_sql                 |4287     |82.2     |
|etl_de_python              |4287     |82.2     |
|etl_de_aws_glue            |4287     |82.2     |
|etl_de_databricks          |4287     |82.2     |
|etl_de_airflow             |4287     |82.2     |
|ia_barreira_alucinacao     |4241     |81.3     |
+---------------------------+---------+---------+
only showing top 15 rows



In [7]:
colunas_gestor = REGRAS_DE_NULO["gestor"][:5]  # amostra
print("Amostra de colunas condicionadas a 'atua_como_gestor = true':")
relatorio.filter(relatorio.coluna.isin(colunas_gestor)).show(truncate=False)

Amostra de colunas condicionadas a 'atua_como_gestor = true':


+-----------------------------+---------+---------+
|coluna                       |qtd_nulos|pct_nulos|
+-----------------------------+---------+---------+
|ia_uso_gestor_descentralizado|4215     |80.8     |
|ia_uso_gestor_centralizado   |4215     |80.8     |
|ia_uso_gestor_copilots       |4215     |80.8     |
|ia_uso_gestor_prod_externo   |4215     |80.8     |
|ia_prioridade_gestor         |4170     |80.0     |
+-----------------------------+---------+---------+



## 6. Grava a Silver

No Glue, essa escrita também catalogaria `silver_state_of_data_2024_2025` no Glue Data Catalog.

In [8]:
df.write.mode("overwrite").parquet(SILVER_PATH)
print("Silver gravada em:", SILVER_PATH)

df_check = spark.read.parquet(SILVER_PATH)
print("Conferência -> linhas:", df_check.count(), "| colunas:", len(df_check.columns))

Silver gravada em: ../../data/silver/state_of_data_2024_2025_silver.parquet


Conferência -> linhas: 5215 | colunas: 175


## 7. Teste rápido: registra como view e roda uma query de exemplo

(as queries completas de P1 a P7 estão em `sql/perguntas.sql`)

In [9]:
df_check.createOrReplaceTempView("state_of_data_silver")

spark.sql("""
    SELECT setor, COUNT(*) AS total
    FROM state_of_data_silver
    WHERE setor IS NOT NULL
    GROUP BY setor
    ORDER BY total DESC
""").show(10, truncate=False)

+------------------------------+-----+
|setor                         |total|
+------------------------------+-----+
|Finanças ou Bancos            |1035 |
|Tecnologia/Fábrica de Software|941  |
|Varejo                        |380  |
|Área de Consultoria           |353  |
|Outra Opção                   |332  |
|Indústria                     |328  |
|Área da Saúde                 |195  |
|Educação                      |194  |
|Internet/Ecommerce            |153  |
|Setor Alimentício             |143  |
+------------------------------+-----+
only showing top 10 rows

